# RAG: Wikipedia-based summarizer
This notebook builds a retrieval-augmented summarization system using LangChain + ChromaDB + SentenceTransformers. It fetches a Wikipedia page, chunks it into ~300-word pieces, stores embeddings in ChromaDB, and builds a LangChain `RetrievalQA` pipeline using a local LLM (Ollama) with a Hugging Face summarizer fallback.

## 1. Environment Setup
Install dependencies (uncomment in Colab/VSCode if needed). Note: Ollama requires a local Ollama server. The notebook includes a fallback summarizer using Hugging Face transformers if Ollama is not found.

In [27]:
# Install the packages if you haven't already (Colab / local environment)
# Uncomment the following lines as needed:
# !pip install wikipedia-api sentence-transformers chromadb langchain transformers torch pandas numpy markdown crewai
import sys
print('Python version:', sys.version)
import os
from pathlib import Path
# Use project root directories to store data and outputs
project_root = Path.cwd().parent
data_dir = project_root / 'data'
outputs_dir = project_root / 'outputs'
data_dir.mkdir(parents=True, exist_ok=True)
outputs_dir.mkdir(parents=True, exist_ok=True)
print('Created directories:', data_dir, outputs_dir)

Python version: 3.10.19 | packaged by conda-forge | (main, Oct 22 2025, 22:46:49) [Clang 19.1.7 ]
Created directories: /Users/gabrielsaco/Documents/GitHub/rag_wikipedia-lab/data /Users/gabrielsaco/Documents/GitHub/rag_wikipedia-lab/outputs


In [28]:
# Install & import checks: run this cell after you created the `rag` environment\n# If you're in Colab, use %pip install commands; otherwise use conda / pip to install packages in your env\n# For convenience (uncomment in Colab):\n# %pip install wikipedia-api sentence-transformers chromadb langchain transformers torch pandas numpy markdown crewai\n\nfrom importlib import util, import_module\nfrom pprint import pprint\n\n# modules to check (import names, not pip names)\nrequired = [\n    'wikipediaapi',\n    'sentence_transformers',\n    'chromadb',\n    'langchain',\n    'transformers',\n    'torch',\n    'pandas',\n    'numpy',\n    'markdown',\n]\n\nmissing = []\nfor pkg in required:\n    try:\n        import_module(pkg)\n        print(pkg, 'OK')\n    except Exception as e:\n        print(pkg, 'MISSING (import failed):', str(e))\n        missing.append(pkg)\n\n# Check langchain vectorstores availability (versions vary across langchain releases)\ntry:\n    from langchain.vectorstores import Chroma as LCChroma\n    print('langchain.vectorstores.Chroma available')\nexcept Exception as e:\n    print('langchain.vectorstores.Chroma not available (LangChain API version mismatch). You can use a custom Chroma retriever or install a compatible LangChain release. Error:', str(e))\n\n# Optional: check whether Ollama client is configured for local LLMs; if not installed, notebook will fall back to HF summarizer\ntry:\n    from langchain.llms import Ollama\n    print('Ollama LLM support available (try to run a local Ollama server if you want to use it)')\nexcept Exception as e:\n    print('Ollama (LangChain integration) not available; notebook will use HF summarizer fallback. Error:', e)\n\nif missing:\n    print('\nMissing packages detected:')\n    pprint(missing)\n    print('\nInstall with conda/pip, e.g.:')\n    print('conda activate rag')\n    print('pip install --upgrade ' + ' '.join(missing))\nelse:\n    print('\nAll key packages are present.')

## 2. Download Wikipedia pages & extract text
We fetch `Federated_learning` and create text chunks. (This cell uses `wikipedia-api`.)

In [29]:
import wikipediaapi
# Use a descriptive user_agent per Wikipedia policy: https://meta.wikimedia.org/wiki/User-Agent_policy
wiki = wikipediaapi.Wikipedia(user_agent='rag_wikipedia_lab/1.0 (gsaco@example.com)', language='en')
# Example pages to fetch and build a small demo corpus
titles = ['Federated_learning', 'Differential_privacy']
pages = {}
for t in titles:
    p = wiki.page(t)
    if not p.exists():
        print('Missing page', t)
    else:
        pages[t] = {'title': p.title, 'text': p.text}
len(pages)

2

## 3. Text cleaning and ~300-Word Chunking
We'll implement a `chunk_text` helper with overlap and return chunks as dictionaries for saving to CSV.

In [30]:
import re
from typing import List, Dict
def chunk_text(text: str, chunk_size=300, overlap=50) -> List[Dict]:
    words = re.split(r'\s+', text.strip())
    chunks = []
    i = 0
    idx = 0
    while i < len(words):
        chunk_words = words[i:i+chunk_size]
        chunk_text = ' '.join(chunk_words).strip()
        if chunk_text:
            chunks.append({'id': None, 'title': None, 'text': chunk_text})
        i += chunk_size - overlap
        idx += 1
    return chunks
# Build dataset
corpus = []
for title, data in pages.items():
    text = data.get('text', '')
    if not text:
        continue
    title_chunks = chunk_text(text, chunk_size=300, overlap=50)
    for i, c in enumerate(title_chunks):
        c['title'] = data['title']
        c['id'] = f'{title}_chunk_{i}'
        corpus.append(c)
len(corpus)

34

## 4. Save Corpus to `/data/wiki_corpus.csv`
Save corpus to CSV with columns `id, title, text`.

In [31]:
import pandas as pd
df = pd.DataFrame(corpus)
# CSV save path
csv_path = str(data_dir / 'wiki_corpus.csv')
df.to_csv(csv_path, index=False)
print('Saved', csv_path, 'with', len(df), 'rows')

Saved /Users/gabrielsaco/Documents/GitHub/rag_wikipedia-lab/data/wiki_corpus.csv with 34 rows


## 5. Initialize Embedding Model and ChromaDB
Load SentenceTransformer and a local Chroma collection. We'll compute embeddings then upsert.

In [32]:
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
model = SentenceTransformer('all-MiniLM-L6-v2')
print('Loaded embedding model on device:', model.device)
# Create local chroma client with persistence (modern config)
client = chromadb.Client(Settings(persist_directory=str(data_dir / 'chromapersist')) )
collection_name = 'wiki_ai'
try:
    collection = client.get_collection(name=collection_name)
    print('Loaded existing collection', collection_name)
except Exception:
    collection = client.create_collection(name=collection_name)
    print('Created collection', collection_name)

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 59282a89-0855-4021-b338-617d82b7281f)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].
Retrying in 1s [Retry 1/5].


Loaded embedding model on device: mps:0
Loaded existing collection wiki_ai


## 6. Embed chunks & upsert to ChromaDB
This computes embeddings in batches and stores documents + metadata to Chroma.

In [33]:
texts = df['text'].tolist()
ids = df['id'].tolist()
metadatas = df[['title']].to_dict(orient='records')
# Compute embeddings in a simple batch loop
batch_size = 64
embeddings = []
for i in range(0, len(texts), batch_size):
    batch = texts[i:i+batch_size]
    emb = model.encode(batch, show_progress_bar=False, convert_to_numpy=True)
    embeddings.extend(emb)
# Add to chroma
collection.add(ids=ids, documents=texts, metadatas=metadatas, embeddings=embeddings)
print('Upserted', len(ids), 'records to Chroma collection')

Upserted 34 records to Chroma collection


## 7. Wrap Chroma with LangChain VectorStore & Retriever
This allows LangChain to call on the vector store as a retriever.

In [34]:
# Try to create a LangChain Chroma vectorstore; if not available, use a simple chroma-based retriever
import langchain
langchain.verbose = False
try:
    from langchain.vectorstores import Chroma as LCChroma
    from langchain.embeddings import SentenceTransformerEmbeddings
    embedding_model = SentenceTransformerEmbeddings(model_name='all-MiniLM-L6-v2')
    # Use LangChain's Chroma wrapper to connect to the local ChromaDB via the chromadb client we created
    vectorstore = LCChroma(collection_name='wiki_ai', embedding_function=embedding_model, chromadb_client=client)
    retriever = vectorstore.as_retriever(search_type='similarity', search_kwargs={'k': 5})
    print('Using LangChain Chroma vectorstore (client)')
except Exception as e:
    print('LangChain Chroma not available; using custom Chroma retriever. Error:', e)
    # A minimal retriever that wraps chromadb collection.query() and returns simple objects similar to LangChain docs
    class SimpleDoc:
        def __init__(self, page_content, metadata):
            self.page_content = page_content
            self.metadata = metadata
    class SimpleRetriever:
        def __init__(self, collection):
            self.collection = collection
        def get_relevant_documents(self, query, k=5):
            results = self.collection.query(query_texts=[query], n_results=k, include=['metadatas','documents'])
            docs = []
            if results and 'documents' in results and results['documents'] and results['documents'][0]:
                for doc_text, meta in zip(results['documents'][0], results.get('metadatas', [])[0]):
                    docs.append(SimpleDoc(page_content=doc_text, metadata=meta))
            return docs
    retriever = SimpleRetriever(collection)
    print('Using custom SimpleRetriever with Chroma collection')

LangChain Chroma not available; using custom Chroma retriever. Error: Chroma.__init__() got an unexpected keyword argument 'chromadb_client'
Using custom SimpleRetriever with Chroma collection


## 8. Configure Local LLM (Ollama) & Create RetrievalQA Chain
Attempt to use Ollama if installed. Otherwise, fallback to a Hugging Face summarizer.

In [35]:
# Create an LLM-based QA chain. We'll try to use LangChain/Ollama if available, otherwise we fall back to a HF summarizer.
import langchain
langchain.verbose = False
llm = None
qa = None
try:
    from langchain.chains import RetrievalQA
    from langchain.llms import Ollama
    # Try to initialize a local Ollama model if accessible
    llm = Ollama(model='mistral')
    qa = RetrievalQA.from_chain_type(llm=llm, chain_type='stuff', retriever=retriever)
    print('Using LangChain RetrievalQA with Ollama')
except Exception as e:
    # Fallback to the existing HF summarizer-based retriever wrapper
    print('LangChain or Ollama not available, using fallback summarizer. Error:', e)
    from transformers import pipeline
    hf_summarizer = pipeline('summarization', model='sshleifer/distilbart-cnn-12-6')
    def simple_rag_query(query):
        docs = retriever.get_relevant_documents(query)
        context = '\n\n'.join([d.page_content for d in docs])
        # Truncate to a reasonable token/word budget if the context is huge
        if len(context.split()) > 3000:
            context = ' '.join(context.split()[:3000])
        s = hf_summarizer(context, max_length=400, min_length=150, do_sample=False)[0]['summary_text']
        return s
    qa = type('SimpleQA', (), {'run': staticmethod(simple_rag_query)})
    print('Using fallback QA (HF summarizer)')

LangChain or Ollama not available, using fallback summarizer. Error: 1 validation error for RetrievalQA
retriever
  value is not a valid dict (type=type_error.dict)


Device set to use mps:0


Using fallback QA (HF summarizer)


## 9. Run Queries and Save Retrieval Examples
Run example queries and save retrieval results to JSON.

In [36]:
import json
examples = []
queries = [
    'Explain federated learning challenges in healthcare.',
    'How does differential privacy help with federated learning?',
]
for q in queries:
    if hasattr(qa, 'run'):
        answer = qa.run(q)
    else:
        # fallback to calling the retriever and HF summarizer wrapper
        answer = qa.run(q)
    docs = retriever.get_relevant_documents(q)
    retrieved = [{'id': getattr(d, 'metadata', {}).get('id', None) or getattr(d, 'id', None), 'title': getattr(d, 'metadata', {}).get('title', None), 'text_snippet': d.page_content[:300]} for d in docs]
    examples.append({'query': q, 'answer': answer, 'retrieved': retrieved})
json_path = str(outputs_dir / 'retrieval_examples.json')
with open(json_path, 'w') as fh:
    json.dump(examples, fh, indent=2)
print('Saved retrieval examples to', json_path)

Token indices sequence length is longer than the specified maximum sequence length for this model (1886 > 1024). Running this sequence through the model will result in indexing errors


Saved retrieval examples to /Users/gabrielsaco/Documents/GitHub/rag_wikipedia-lab/outputs/retrieval_examples.json


## 10. Generate and Save 400–500 Word RAG Summary
Retrieve top chunks for a target query, combine them, and ask the LLM to generate a coherent 400–500 word summary.

In [37]:
final_query = 'Summarize the challenges, technical and policy, of using federated learning in healthcare in 400-500 words.'
# Use LangChain RetrievalQA if available; otherwise perform multi-stage summarization using HF summarizer.
try:
    if qa is not None and hasattr(qa, 'llm') and qa.llm is not None:
        summary = qa.run(final_query)
    else:
        raise Exception('No langchain QA available; using HF fallback')
except Exception as e:
    print('Falling back to a multi-stage HF summarizer approach; reason:', e)
    from transformers import pipeline
    try:
        summarizer = pipeline('summarization', model='facebook/bart-large-cnn')
    except Exception:
        summarizer = pipeline('summarization', model='sshleifer/distilbart-cnn-12-6')
    # Retrieve top docs and produce small batch summaries (map step)
    top_docs = retriever.get_relevant_documents(final_query)[:12]
    group_size = 3
    group_summaries = []
    for i in range(0, len(top_docs), group_size):
        group = top_docs[i:i+group_size]
        text = '\n\n'.join([d.page_content for d in group])
        if len(text.split()) > 2000:
            text = ' '.join(text.split()[:2000])
        s = summarizer(text, max_length=200, min_length=100, do_sample=False)[0]['summary_text']
        group_summaries.append(s)
    # Expand each group by adding top doc sentences (low-risk factual expansion)
    import re
    expanded_paragraphs = []
    for i in range(len(group_summaries)):
        skeleton = group_summaries[i]
        # gather sentences from the docs for this group (avoid instructions in prompts)
        docs_for_group = top_docs[i*group_size:(i+1)*group_size]
        key_sents = []
        for d in docs_for_group:
            sents = re.split(r'(?<=[.!?])\s+', d.page_content.strip())
            for s in sents[:3]:
                if len(s.strip()) > 40 and s.strip() not in key_sents:
                    key_sents.append(s.strip())
        # Merge skeleton and supporting sentences into paragraph
        paragraph = skeleton.strip() + '\n\n' + ' '.join(key_sents[:3])
        expanded_paragraphs.append(paragraph)
    combined = '\n\n'.join(expanded_paragraphs)
    # If combined is too short, append more factual sentences from remaining docs
    words = combined.split()
    extra_idx = 0
    while len(words) < 400 and extra_idx < len(top_docs):
        more_sents = re.split(r'(?<=[.!?])\s+', top_docs[extra_idx].page_content.strip())
        for s in more_sents[:4]:
            if len(words) >= 400:
                break
            text_to_add = s.strip()
            if text_to_add not in combined and len(text_to_add) > 40:
                combined += '\n\n' + text_to_add
                words = combined.split()
        extra_idx += 1
    summary = combined
print('Summary length (words):', len(summary.split()))
# Save to markdown file
summary_path = str(outputs_dir / 'rag_summary.md')
with open(summary_path, 'w') as fh:
    fh.write('# RAG Summary — Federated Learning in Healthcare\n\n')
    fh.write(summary)
print('Saved summary to', summary_path)

Falling back to a multi-stage HF summarizer approach; reason: No langchain QA available; using HF fallback


Device set to use mps:0


Summary length (words): 407
Saved summary to /Users/gabrielsaco/Documents/GitHub/rag_wikipedia-lab/outputs/rag_summary.md


## 11. Quick verification and counts
Check the CSV, number of chunks, and Chroma collection size.

In [38]:
import os
csv_path = str(data_dir / 'wiki_corpus.csv')
assert os.path.exists(csv_path), f'CSV not found: {csv_path}'
print('CSV rows:', len(pd.read_csv(csv_path)))
# chroma count — check if the collection has documents (API varies)
try:
    print('Chroma collection count (approx):', len(collection.get(ids=ids)))
except Exception as e:
    print('Could not query collection count; error', e)

CSV rows: 34
Chroma collection count (approx): 7


## 12. Bonus: Conceptual comparison + write reflection.md
Write a short reflection contrasting the multi-agent workflow vs RAG. Save to `/outputs/reflection.md`.

In [39]:
reflection = '''# Reflection: Multi-agent vs RAG\n\nMulti-agent systems use several cooperating agents to break down a task, which helps handle ambiguity by distributing subquestions but may propagate contradictions across agents when they synthesize their answers. In contrast, RAG relies on concrete retrieved evidence and a single summarizer which encourages factuality but is sensitive to retrieval coverage.\n\nMulti-agent pros: good for open-ended creative workflows; handles diverse perspectives; can propose multiple strategies.\nMulti-agent cons: needs orchestration and stronger validation to avoid contradiction.\n\nRAG pros: better suited for factual questions where grounding to retrieved sources reduces hallucination; easier to validate by inspecting retrievals.\nRAG cons: limited by the retrieval coverage and embedding model; does not create new knowledge beyond the corpus.\n\nWhich approach when? For open-ended ideation and exploration, a multi-agent approach can help synthesize varied viewpoints. For fact-dense, evidence-sensitive problems (like clinical or legal questions), RAG is preferable because it provides explicit supporting context.\n'''
# Save to file
with open(str(outputs_dir / 'reflection.md'), 'w') as fh:
    fh.write(reflection)
print('Saved reflection to /outputs/reflection.md')

Saved reflection to /outputs/reflection.md
